# Craig Transform

Transforming $G_{UV},\,G_{NE}$ to $g_{D}$

The Craig transform provides functionality to generate gravity from differential curvatures.

This tutorial demonstrates its use on publicly available data (resources.vic.gov.au) from the Fosterville Falcon airborne gravity gradiometer survey.

The Craig transform calculates any or all components of the gravity gradient and the gravity acceleration from measured differential curvature gradients. The transform is performed in the wavenumber domain using the Fourier transform. The gravity gradient tensor is derived from:

$$\mathcal{F}\left(\Gamma\right)=\frac{2}{\left(k_{N}+i k_{E}\right)^{2}}\begin{bmatrix}k_N^2 & k_N k_E & ik_N k \\ & k_E^2 & i k_E k \\  &  & -k^2 \\\end{bmatrix}\mathcal{F}\left(\Gamma_{UV}+i \Gamma_{NE}\right)$$

and the gravitational acceleration vector from

$$\mathcal{F}\left(\mathbf{g}\right)=\frac{2}{\left(k_{N}+i k_{E}\right)^{2}}\begin{bmatrix}-ik_N & -ik_E & -k \end{bmatrix}\mathcal{F}\left(\Gamma_{UV}+i \Gamma_{NE}\right)$$

where $k=\sqrt{k_N^2+k_E^2}$ is the radial wavenumber on the horizontal plane.

The implementation here only derives the vertical gravitational acceleration:

$$\mathcal{F}\left(g_D\right)=\frac{-2k}{\left(k_{N}+i k_{E}\right)^{2}}\mathcal{F}\left(\Gamma_{UV}+i \Gamma_{NE}\right)$$

The method proceeds in several steps.

1. The measured differential curvature ($\Gamma_{UV}$ and $\Gamma_{NE}$) data are separately gridded. This is done by minimum curvature with a default cell size 1/4 of the line spacing although the user may select their own value. Grid cells further than 3 cells from a located data value are marked "missing".

1. Each grid is padded to extend the size of the grid in all four directions, and to fill any missing data. The padding may be done by sampling from a regional gravity grid, by filling with the mean value, or by a constant value. Regional gravity padding provides the best result. The regional grid is transformed by the reverse Craig transform to the differential curvature components and interpolated onto the locations of the measured differential curvature grids.

1. The padded data are transformed to $g_D$ using the bottom equation above. Then the cells that were padded for the transform are marked as missing since they did not originally have a measured value. The grid is then trimmed to the smallest rectangle possible without removing good data.

1. This grid is sampled back to the `whizzFile` from which the differential curvature data originated.

Note that the method treats all data as though it is located on a horizontal plane. This is a good approximation for many airborne gravity gradient surveys but will give inaccurate results for surveys draped to follow rugged terrain.

## Fosterville Example

The Fosterville data were downloaded from Resources Victoria. The located data were in ASEG-GDF2 format and these were written to `whizzFile` in HDF5 format. Most channels were omitted to reduce the data volume. It is this minimalist data set that is used here. For no particular reason, this example uses the free-air data.

___
Import the required modules, and set the path to the geowhizz files.

In [ ]:
from pathlib import Path
import pegasusQC as qc

data_root = r'./FostervilleData/'
FostervilleHDF_file = Path(data_root + r'Foster_craig.hdf5')
regional_fa_ers = Path(data_root + r'Gravmap2019-grid-grv_fa.ers')

In [ ]:
if not FostervilleHDF_file.exists():
    print("Ask Mark for help!!!")

In [ ]:
# Find the channel names for the free-air Gne and Guv data
qc.reportChannels(FostervilleHDF_file)

In [ ]:
gD_grid = qc.craig_transform(
    whizzFile=FostervilleHDF_file, 
    gne_chan='Falc_A_NE_0p00_0p18Hz_MeanLev', 
    guv_chan='Falc_A_UV_0p00_0p18Hz_MeanLev',
    gd_chan=None,
    result_units='um/s/s',
    mask_polygon=None,
    pad_cells=256, 
    padding_mode="regional", 
    regional_grid_file=regional_fa_ers,
    regional_grav_units='gu',
)

## Telfer Example

These data were downloaded from the Geological Survey of Western Australia at https://geodownloads.dmp.wa.gov.au/downloads/geophysics/71234/.

In [ ]:
from pathlib import Path
import pegasusQC as qc

data_root = r'./TelferData/'
TelferHDF_file = Path(data_root + r'Telfer_craig.hdf5')

regional_fa_ers = Path(data_root + r'Gravmap2019-grid-grv_dtgir.ers')

true_gd_ers = Path(data_root + r'2131_1_Fourier_gD_2p2_conformed_final.ers')

In [ ]:
if not TelferHDF_file.exists():
    print("Ask Mark for help!!!")
    # %run ./Prepare_EastVicData.ipynb

In [ ]:
# Find the channel names for the free-air Gne and Guv data
qc.reportChannels(TelferHDF_file)

In [ ]:
gD_grid = qc.craig_transform(
    whizzFile=TelferHDF_file,
    gne_chan='A_NE_2p67',
    guv_chan='A_UV_2p67',
    gd_chan=None,
    cell_size=80.0,
    result_units='um/s/s',
    mask_polygon=None,
    pad_cells=256,
    padding_mode="regional",
    regional_grid_file=regional_fa_ers,
    regional_grav_units='um/s/s'
)

In [ ]:
reg_bg, _ = qc.gridfile_to_xa(regional_fa_ers)
qc.xdImage(reg_bg, 'regional BG [um/s/s]')

In [ ]:
true_bg, _ = qc.gridfile_to_xa(true_gd_ers)
qc.xdImage(true_bg, 'true 2p2 BG [mGal]', minClip=-20, maxClip=5)

In [ ]:
qc.xdImage(gD_grid, 'derived BG', hs=True, minClip=-120, maxClip=75)